# Probe Guidance: World Modeling

In [1]:
import sys
sys.path.insert(0, "/home/jack/code/vjepa2-probe-guidance/vjepa2")
print(sys.path)

['/home/jack/code/vjepa2-probe-guidance/vjepa2', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages', '/home/jack/code/vjepa2-probe-guidance/vjepa2/src', '/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages/rerun_sdk']


In [2]:
from pathlib import Path
import copy
import os

import numpy as np
import torch
from torch.nn import functional as F
import torchvision.transforms as T
from scipy.spatial.transform import Rotation
from tqdm import tqdm

In [3]:
from app.vjepa_ll_probe_guidance.ll_probe_guidance import LLProbeGuidanceDataset, standardize_actions, standardize_states
from app.vjepa_ll_probe_guidance.utils import init_video_model
from app.vjepa_ll_probe_guidance.transforms import make_transforms

/home/jack/code/vjepa2-probe-guidance/.venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [4]:
def l1(a, b):
    return torch.mean(torch.abs(a - b), dim=-1)


def round_small_elements(tensor, threshold):
    mask = torch.abs(tensor) < threshold
    new_tensor = tensor.clone()
    new_tensor[mask] = 0
    return new_tensor


def cem(
    context_frame,
    context_pose,
    goal_frame,
    world_model,
    rollout=1,
    cem_steps=100,
    momentum_mean=0.25,
    momentum_std=0.95,
    samples=100,
    topk=10,
    verbose=False,
    maxnorm=0.05,
    axis={},
    objective=l1,
    device="cpu"
):
    """
    :param context_frame: [B=1, T=1, HW, D]
    :param goal_frame: [B=1, T=1, HW, D]
    :param world_model: f(context_frame, action) -> next_frame [B, 1, HW, D]
    :return: [B=1, rollout, 7] an action trajectory over rollout horizon

    Cross-Entropy Method
    -----------------------
    1. for rollout horizon:
    1.1. sample several actions
    1.2. compute next states using WM
    3. compute similarity of final states to goal_frames
    4. select topk samples and update mean and std using topk action trajs
    5. choose final action to be mean of distribution
    """
    context_frame = context_frame.repeat(samples, 1, 1, 1)  # Reshape to [S, 1, HW, D]
    goal_frame = goal_frame.repeat(samples, 1, 1, 1)  # Reshape to [S, 1, HW, D]
    context_pose = context_pose.repeat(samples, 1, 1)  # Reshape to [S, 1, 7]

    # Current estimate of the mean/std of distribution over action trajectories
    # Start with a normal distribution (mean=0 and std=1)
    mean = torch.zeros((rollout, 3), device=device)
    std = torch.ones((rollout, 3), device=device) * maxnorm

    # NOTE: What the hell is this for?
    for ax in axis.keys():
        mean[:, ax] = axis[ax]

    def sample_action_traj():
        """Sample several action trajectories"""
        action_traj, frame_traj, pose_traj = None, context_frame, context_pose

        for h in range(rollout):

            # -- sample new action
            # NOTE: why do * std[h] and + mean[h]? Does that effectively change the distribution?
            # NOTE: furthermore, torch.randn samples from a normal distribution...
            action_samples = torch.randn(samples, mean.size(1), device=device) * std[h] + mean[h]
            action_samples[:, :3] = torch.clip(action_samples[:, :3], min=-maxnorm, max=maxnorm)

            # NOTE: same here, what is axis doing?
            for ax in axis.keys():
                action_samples[:, ax] = axis[ax]

                
            action_samples = torch.cat(
                [
                    action_samples[:, :3],
                    # NOTE: This is ignoring rotation from my understanding, undo this???
                    torch.zeros((len(action_samples), 3), device=device),
                ],
                dim=-1,
            )[:, None]

            action_traj = (
                torch.cat([action_traj, action_samples], dim=1) if action_traj is not None else action_samples
            )

            # -- compute next state
            next_frame, next_pose = world_model(frame_traj, action_traj, pose_traj)
            frame_traj = torch.cat([frame_traj, next_frame], dim=1)
            pose_traj = torch.cat([pose_traj, next_pose], dim=1)

        return action_traj, frame_traj

    def select_topk_action_traj(final_state, goal_state, actions):
        """Get the topk action trajectories that bring us closest to goal"""
        sims = objective(final_state.flatten(1), goal_state.flatten(1))
        indices = sims.topk(topk, largest=False).indices
        selected_actions = actions[indices]
        return selected_actions

    for step in tqdm(range(cem_steps), disable=True):
        action_traj, frame_traj = sample_action_traj()
        selected_actions = select_topk_action_traj(
            final_state=frame_traj[:, -1], goal_state=goal_frame, actions=action_traj
        )
        mean_selected_actions = selected_actions.mean(dim=0)
        std_selected_actions = selected_actions.std(dim=0)

        # -- Update new sampling mean and std based on the top-k samples
        mean = torch.cat(
            [
                mean_selected_actions[..., :3] * (1.0 - momentum_mean) + mean[..., :3] * momentum_mean,
            ],
            dim=-1,
        )
        std = torch.cat(
            [
                std_selected_actions[..., :3] * (1.0 - momentum_std) + std[..., :3] * momentum_std,
            ],
            dim=-1,
        )

        print(f"new mean: {mean.sum(dim=0)} {std.sum(dim=0)}")

    new_action = torch.cat(
        [
            mean[..., :3],
            torch.zeros((rollout, 3), device=device),
        ],
        dim=-1,
    )[None, :]

    return new_action

In [5]:
def forward_target(c, normalize_reps=True):
    B, C, T, H, W = c.size()
    c = c.permute(0, 2, 1, 3, 4).flatten(0, 1).unsqueeze(2).repeat(1, 1, 2, 1, 1)
    h = encoder(c)
    h = h.view(B, T, -1, h.size(-1)).flatten(1, 2)
    if normalize_reps:
        h = F.layer_norm(h, (h.size(-1),))
    return h

In [6]:
# NOTE: the actions are in the probe's coordinate frame, while the states are 
# in the camera's coordinate frame. Does this have a negative impact on the model?
# Nonetheless, computing the new pose needs to account for this discrepancy.

# NOTE: state and action may be standardized, may need to account for this or rethink dataset/dataloader
# to not do standardization. Perhaps standardization should happen in training/testing and before passing to model
# forward functions (or within the forward functions themselves). doing it in dataset makes it hard to do visualizations.
# breaking standardization out can be good from a single-responsibility principle standpoint.

def compute_new_pose(pose, action):
    """
    :param pose: [B, T=1, 6] (x, y, z, rx, ry, rz in Euler angles) in camera frame
    :param action: [B, T=1, 6] (dx, dy, dz, drx, dry, drz) in local object frame
    :returns: [B, T=1, 6] updated pose in camera frame
    """
    device, dtype = pose.device, pose.dtype
    
    pose_np = pose[:, 0].detach().cpu().numpy()
    action_np = action[:, 0].detach().cpu().numpy()

    R_pose = Rotation.from_euler("xyz", pose_np[:, 3:6], degrees=True)
    R_pose_mat = R_pose.as_matrix()  # [B, 3, 3]

    # local action translation: [B, 3, 1] column vector
    local_dxyz = action_np[:, :3, None]
    
    global_dxyz = (R_pose_mat @ local_dxyz).squeeze(-1)  # [B, 3]
    new_xyz = pose_np[:, :3] + global_dxyz

    R_action = Rotation.from_euler("xyz", action_np[:, 3:6], degrees=True)
    R_new = R_pose * R_action

    new_angle = R_new.as_euler("xyz", degrees=True)  # [B, 3]

    new_pose = np.concatenate([new_xyz, new_angle], axis=-1)

    # Restore dimensions to [B, T=1, 6]
    return torch.from_numpy(new_pose).to(device=device, dtype=dtype)[:, None]

In [7]:
class WorldModel(object):

    def __init__(
        self,
        encoder,
        predictor,
        tokens_per_frame,
        transform,
        mpc_args={
            "rollout": 2,
            "samples": 400,
            "topk": 10,
            "cem_steps": 10,
            "momentum_mean": 0.15,
            "momentum_std": 0.15,
            "maxnorm": 0.05,
            "verbose": True,
        },
        normalize_reps=True,
        device="cpu",
    ):
        super().__init__()
        self.encoder = encoder
        self.predictor = predictor
        self.normalize_reps = normalize_reps
        self.transform = transform
        self.tokens_per_frame = tokens_per_frame
        self.device = device
        self.mpc_args = mpc_args

    def encode(self, image):
        clip = np.expand_dims(image, axis=0)
        clip = self.transform(clip)[None, :]
        B, C, T, H, W = clip.size()
        clip = clip.permute(0, 2, 1, 3, 4).flatten(0, 1).unsqueeze(2).repeat(1, 1, 2, 1, 1)
        clip = clip.to(self.device, non_blocking=True)
        h = self.encoder(clip)
        h = h.view(B, T, -1, h.size(-1)).flatten(1, 2)
        if self.normalize_reps:
            h = F.layer_norm(h, (h.size(-1),))
        return h

    def infer_next_action(self, rep, pose, goal_rep, close_gripper=None):

        def step_predictor(reps, actions, poses):
            B, T, N_T, D = reps.size()
            reps = reps.flatten(1, 2)
            standardized_poses = standardize_states(poses)
            standardized_actions = standardize_actions(actions)
            next_rep = self.predictor(reps, standardized_actions, standardized_poses)[:, -self.tokens_per_frame :]
            if self.normalize_reps:
                next_rep = F.layer_norm(next_rep, (next_rep.size(-1),))
            next_rep = next_rep.view(B, 1, N_T, D)
            next_pose = compute_new_pose(poses[:, -1:], actions[:, -1:])
            return next_rep, next_pose

        mpc_action = cem(
            context_frame=rep,
            context_pose=pose,
            goal_frame=goal_rep,
            world_model=step_predictor,
            device=self.device,
            **self.mpc_args,
        )[0]

        return mpc_action

## Initialization
Initialize the encoder, predictor, dataset, and other key parameters

In [8]:
device = "cpu"

In [9]:
encoder, predictor = init_video_model(
        device=device,
        patch_size=16,
        max_num_frames=512,
        tubelet_size=2,
        model_name="vit_large",
        crop_size=256,
        pred_depth=12,
        pred_num_heads=12,
        pred_embed_dim=768,
        action_embed_dim=6,
        predictor_type="ac",
        pred_is_frame_causal=True,
        use_extrinsics=False,
        use_sdpa=True,
        use_rope=True
    )
target_encoder = copy.deepcopy(encoder)

encoder.eval()
predictor.eval()
target_encoder.eval()

def load_state_dict_with_ddp_fix(model, state_dict):
    new_state_dict = {}
    for k, v in state_dict.items():
        # Remove 'module.' prefix if it exists
        new_key = k.replace("module.", "")
        new_state_dict[new_key] = v

    model.load_state_dict(new_state_dict, strict=True)
    return model

resume_path = os.path.join("/home/jack/code/vjepa2-probe-guidance/vjepa2/outputs/ll_probe_guidance_vitl_4", "best.pt")
if os.path.exists(resume_path):
    print(f"Loading checkpoint from {resume_path}")
    checkpoint = torch.load(resume_path, map_location=torch.device("cpu"))
    encoder = load_state_dict_with_ddp_fix(encoder, checkpoint["encoder"])
    predictor = load_state_dict_with_ddp_fix(predictor, checkpoint["predictor"])
    target_encoder = load_state_dict_with_ddp_fix(target_encoder, checkpoint["target_encoder"])
else:
    print(f"Checkpoint not found at {resume_path}")

print("=" * 20 + "PREDICTOR" + "=" * 20)
print(predictor)
print("=" * 20 + "TARGET ENCODER" + "=" * 20)
print(target_encoder)

Loading checkpoint from /home/jack/code/vjepa2-probe-guidance/vjepa2/outputs/ll_probe_guidance_vitl_4/best.pt
====================PREDICTOR====================
VisionTransformerPredictorAC(
  (predictor_embed): Linear(in_features=1024, out_features=768, bias=True)
  (action_encoder): Linear(in_features=6, out_features=768, bias=True)
  (state_encoder): Linear(in_features=6, out_features=768, bias=True)
  (extrinsics_encoder): Linear(in_features=5, out_features=768, bias=True)
  (predictor_blocks): ModuleList(
    (0-11): 12 x ACBlock(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
      (attn): ACRoPEAttention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, b

In [10]:
crop_size = 256
tokens_per_frame = int((crop_size // encoder.patch_size) ** 2)
transform = make_transforms(
    crop_size=crop_size,
)

In [11]:
def load_clips(sample, device):
    clips = sample[0].to(device, non_blocking=True)  # [B C T H W]
    actions = sample[1]  # [B T-1 6]
    states = sample[2]  # [B T 6]
    extrinsics = sample[3].to(device, dtype=torch.float, non_blocking=True)  # [B T 6]
    return (clips, actions, states, extrinsics)

In [12]:
# SET THIS TO THE NUMBER OF FRAMES YOU WANT IN A CLIP
T = 8

dataset = LLProbeGuidanceDataset(
    data_root="/home/jack/data/probe_guidance_dataset_june/test",
    frames_per_clip=T,
    frame_skip=1,
    frames_per_second=4,
    transform=transform,
    is_train=False,
)

loader = torch.utils.data.DataLoader(
    dataset,
    shuffle=False,
    batch_size=1,
    drop_last=True,
    pin_memory=False,
    num_workers=8,
)

Scanning 7 episodes for valid tracking clips...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 33.27it/s]

Retained 7 episodes.


## Model Predictive Control (MPC)
VJEPA2 uses MPC to perform tasks like controlling a robot arm to accomplish some goal. The "predictor" that we trained is used to understand how actions affect the state of the world. Then, the cross entropy method (CEM) is used to do "optimization", i.e. it will find a set of actions that minimize error/costs. The error in this case would be, "does the VJEPA2 AC predictor think that this action will reduce the L1 distance to the goal state"?. Then it does "receding horizon" control by simply taking the first action in the action trajectory that CEM found.

### MPC Example

In [ ]:
world_model = WorldModel(
    encoder=encoder,
    predictor=predictor,
    tokens_per_frame=tokens_per_frame,
    transform=transform,
    mpc_args={
        "rollout": 1,
        "samples": 50,
        "topk": 10,
        "cem_steps": 50,
        "momentum_mean": 0.15,
        "momentum_std": 0.75,
        "maxnorm": 0.075,
        "verbose": True
    },
    normalize_reps=True,
    device=device
)

sample = next(iter(loader))
clips, actions, states, _ = load_clips(sample, device)
clips = clips[:, :, :T]
actions = actions[:, :T-1]
states = states[:, :T]
print(f"clips: {clips.shape}; states: {states.shape}; actions: {actions.shape}")

with torch.no_grad():
    h = forward_target(clips)
    print(f"{h.shape=}")
    z_n, z_goal = h[:, :tokens_per_frame], h[:, tokens_per_frame:tokens_per_frame*2]
    s_n = states[:, :1]
    print(f"Starting planning using Cross-Entropy Method...")
    wm_actions = world_model.infer_next_action(z_n, s_n, z_goal).cpu().numpy()

print(f"Actions returned by planning with CEM (x,y,z) = ({wm_actions[0, 0]:.5f}, {wm_actions[0, 1]:.5f}, {wm_actions[0, 2]:.5f})")
print(f"Ground truth actions (x,y,z) = ({actions[0, 0, 0]:.5f}, {actions[0, 0, 1]:.5f}, {actions[0, 0, 2]:.5f})")

clips: torch.Size([1, 3, 8, 256, 256]); states: torch.Size([1, 8, 6]); actions: torch.Size([1, 7, 6])


/usr/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


h.shape=torch.Size([1, 2048, 1024])
Starting planning using Cross-Entropy Method...
new mean: tensor([ 0.0465,  0.0095, -0.0502]) tensor([0.0613, 0.0655, 0.0620])
new mean: tensor([ 0.0655, -0.0006, -0.0606]) tensor([0.0491, 0.0556, 0.0509])


## World Model Performance on Test Set

We want to know how well this world modeling approach works from two perspectives: time and error. We need to know how long it takes the world model to do a single inference step, and how much error this is in it's predictions. Besides raw performance of the predictor (we could even assume a perfect predictor), There are several parameters that have control over the timing and error of the world model, namely the number of CEM steps, rollout steps, and samples. To get a good idea of which sets of parameters work best, we will need to do a hyperparamter search over the test set. The best set of parameters should result in short inference times and low error between predicted actions and ground truth actions.